In [ ]:
import pandas as pd
import python_utilities as pytb
import matplotlib.pyplot as plt
import flameplot
import os

label_dict = {"x": "$x$ (m)", "T": "$T$ (K)", "U_0": "$u$ (m/s)", "p": "$p$ (Pa)"}

foam_case = f"temp/test_fpFlame/case.foam"
cantera_reference = f"reference/cantera/fp_Z0.05000.parq"

print(f"Using foam case: {foam_case}.")
print(f"Using cantera reference: {cantera_reference}.")

# Check convergence
- Take the minimum time step, where the results in state-space do not show a large difference.

In [ ]:
# Load data as time-series and check convergence
df = pytb.pyvistaTools.foam_time_series_to_df(foam_case, keep_boundaries=["inlet", "outlet"])
df.sort_values(["t", "x"], inplace=True, ignore_index=True)

yvs = ['Z', 'yc', 'omega_yc', 'U_0', 'rho', 'p'] 
fig, axs = flameplot.create_multiplot_layout(nvars=len(yvs), sharex=True)

xv = "x"
for yv, ax in zip(yvs, axs):
    for t in df["t"].unique()[5:-1]:
        tmp = df[df["t"]==t]
        tmp.plot(xv, yv, ax=ax, legend=False, label=f"t={t}")
    ax.set_xlabel(label_dict.get(xv, xv))
    ax.set_ylabel(label_dict.get(yv, yv))
axs[0].legend()
fig.set_size_inches(12,6)
fig.tight_layout()

# Compare with cantera (if possible)

In [ ]:
result = pytb.pyvistaTools.foam_to_df(foam_case, time=0.005, index=False, case_type="reconstructed", keep_boundaries=["inlet", "outlet"])
result.sort_values("x", inplace=True, ignore_index=True)

try:
    ct_reference = pd.read_parquet(cantera_reference)
except:
    ct_reference = None
    print("Cantera reference not found.")

if ct_reference is not None:
    result["U_shift"] = result["U_0"] - result["U_0"][0]
    ct_reference["U_shift"] = ct_reference["U_0"] - ct_reference["U_0"][0]

    yvs = ['U_shift', 'Z', 'U_0', 'rho', 'p'] 
    xv = "yc"

    fig, axs = flameplot.create_multiplot_layout(nvars=len(yvs), sharex=True)
    for yv, ax in zip(yvs, axs):
        ct_reference.plot(xv, yv, ax=ax, legend=False, label="cantera", color="k")
        result.plot(xv, yv, ax=ax, legend=False, label="foam", linestyle="-.", color="C1")
        ax.set_xlabel(label_dict.get(xv, xv))
        ax.set_ylabel(label_dict.get(yv, yv))
    axs[0].legend()
    fig.set_size_inches(12,6)
    fig.tight_layout()
    fig.savefig("reference/figures/compare_openfoam_cantera.png")

# Export references result

In [ ]:
t = 0.01
result = pytb.pyvistaTools.foam_to_df(foam_case, time=t, index=False, case_type="reconstructed", keep_boundaries=["inlet", "outlet"])
result.sort_values("x", inplace=True, ignore_index=True)
result.to_parquet('reference/foam/t_0.010.parq')

# Create plot for the report

In [ ]:
result = pd.read_parquet(f"reference/foam/t_0.010.parq")
cantera_reference = f"reference/cantera/fp_Z0.05000.parq"

try:
    ct_reference = pd.read_parquet(cantera_reference)
except:
    ct_reference = None
    print("Cantera reference not found.")

if ct_reference is not None:
    result["U_shift"] = result["U_0"] - result["U_0"][0]
    ct_reference["U_shift"] = ct_reference["U_0"] - ct_reference["U_0"][0]

    yvs = ['U_shift', 'Z', 'U_0', 'rho', 'p'] 
    xv = "yc"

    fig, axs = flameplot.create_multiplot_layout(nvars=len(yvs), sharex=True)
    for yv, ax in zip(yvs, axs):
        ct_reference.plot(xv, yv, ax=ax, legend=False, label="cantera", color="k")
        result.plot(xv, yv, ax=ax, legend=False, label="foam", linestyle="-.", color="C1")
        ax.set_xlabel(label_dict.get(xv, xv))
        ax.set_ylabel(label_dict.get(yv, yv))
    axs[0].legend()
    fig.set_size_inches(12,6)
    fig.tight_layout()
    fig.savefig("report/figures/compare_openfoam_cantera.png")